In [1]:
import torch
import numpy
from datasets import load_dataset

In [2]:
print(torch.cuda.is_available())

True


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

# logging.enable_progress_bar()
# logging.set_verbosity_info()

In [5]:
model_name = "Qwen/Qwen3-4B"
local_model_path = "models/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AutoModelForCausalLM.from_pretrained(local_model_path, device_map="auto")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [7]:
messages = [
    {
        "role": "user", "content": "Hello, who are you?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    do_sample=True
)

output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
print("output: ", output)

output:  <|im_start|>user
Hello, who are you?<|im_end|>
<|im_start|>assistant
<think>
Okay, the user asked, "Hello, who are you?" I need to respond appropriately. Let me start by introducing myself clearly. I should mention that I'm Qwen, developed by Alibaba Cloud. It's important to highlight my capabilities, like answering questions, creating content, and providing assistance. I should keep the tone friendly and approachable. Also, I should invite them to ask questions or share what they need help with. Make sure the response is concise but covers all the key points. Let me check if there's anything else I need to include. Maybe mention that I'm here to help with various tasks. Yeah, that should do it.
</think>

Hello! I'm Qwen, a large language model developed by Alibaba Cloud. I can help with answering questions, creating content, and providing assistance with a wide range of tasks. How can I help you today? 😊<|im_end|>


In [29]:
ds = load_dataset("json", data_files="gsm8k_dataset/train.jsonl")
print(ds["train"])

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})


In [30]:
print(ds["train"]["question"][0])
print(ds["train"]["answer"][0])
questions = ds["train"]["question"][:10]
answers = ds["train"]["answer"][:10]

for _ in range(10):
    print(questions[_])
    print("="*60)

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
How many clips did Natalia sell in May? ** Natalia sold 48/2 = <<48/2=24>>24 clips in May.
How many clips did Natalia sell altogether in April and May? ** Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and 

In [32]:
SYSTEM_PROMPT = '''
You are a math problem solver. Solve given problems carefully using step-by-step reasoning.


The final line of your response MUST be exactly in this format:
#### <numeric answer>

where <numeric answer> is a single number (no units, no commas, no extra words). Do not write anything after the #### line.
'''
for i in range(3):
    messages = [
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user", "content": questions[i]
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        do_sample=True
    )
    
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    print("output: ", output)
    print("=" * 80)

output:  <|im_start|>system

You are a math problem solver. Solve given problems carefully using step-by-step reasoning.


The final line of your response MUST be exactly in this format:
#### <numeric answer>

where <numeric answer> is a single number (no units, no commas, no extra words). Do not write anything after the #### line.
<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. Natalia sold clips to 48 friends in April. Then in May, she sold half as many. So I need to find out how many she sold in May first. Half of 48 would be 24, right? Because 48 divided by 2 is 24. Then to find the total for both months, I add April and May together. So 48 plus 24. Let me check that again. 48 plus 24 is 72. So altogether, she sold 72 clips. Wait, is there anything else? The problem says "half as many clips i